# Generate Fine-tuning Data from Source Texts

This notebook implements the complete text collection and section extraction workflow for fine-tuning a writing model.

## 📋 What This Notebook Does

This notebook takes your **selected stylistic inspirations** (from `1_stylistic_inspiration.ipynb`) and transforms them into **ready-to-use fine-tuning data** through these steps:

### 🔄 Complete Workflow

1. **📥 Downloads Source Texts** 
   - Uses AI to find URLs for each work automatically
   - Downloads texts from Project Gutenberg, Archive.org, etc.
   - **Verifies** that downloaded content matches the expected work (author + title)
   - Saves verified texts locally

2. **🔍 Creates Vector Databases**
   - Splits texts into searchable paragraphs
   - Generates embeddings for semantic search
   - Enables finding sections by natural language descriptions

3. **🤖 Gets Section Recommendations**
   - **Step A:** LLM uses its knowledge to identify what types of sections to look for
   - **Step B:** LLM searches the actual text to find matching sections
   - Provides 5-10 recommended sections per work with explanations

4. **✂️ Interactively Selects Sections**
   - Review each recommended section
   - Adjust boundaries (expand/contract)
   - Select which sections to include
   - Navigate between multiple works

5. **💾 Assembles Fine-tuning Data**
   - Saves all selected sections in the format needed for `3_finetune-e2e-fiction.ipynb`
   - Preserves metadata (source work, author, reason for selection)

## 🎯 End Result

A fine-tuning data file (`combined_act2.txt`) containing curated text sections from your selected inspiration sources, ready to use in the next notebook.

---

**Prerequisites:** Run `1_stylistic_inspiration.ipynb` first to select works.

**Next Step:** Use the output file in `3_finetune-e2e-fiction.ipynb` to train your model.


---

## ⚙️ STEP 1: Configuration & Setup

Configure project settings, API clients, and file paths.


In [10]:

import os
import json
import re
import requests
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from openai import OpenAI
from typing import Optional, List, Dict
from tqdm import tqdm
import chromadb
from chromadb.config import Settings
import hashlib

PROJECT_NAME = "QoGD"

# Model configuration
MODEL_NAME = "gpt-5"

# Setup client
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# File paths
SOURCE_PROMPTS_DIR = f"source_prompts/{PROJECT_NAME}/"
SOURCE_TEXTS_DIR = f"source_texts/{PROJECT_NAME}/"
FINETUNING_DATA_DIR = f"finetuning_data/{PROJECT_NAME}/"
SELECTED_INSPIRATIONS_JSON = f"{SOURCE_PROMPTS_DIR}selected_inspirations.json"
OUTPUT_DATA_FILE = f"{FINETUNING_DATA_DIR}combined_act2.txt"

# Ensure directories exist
os.makedirs(SOURCE_TEXTS_DIR, exist_ok=True)
os.makedirs(FINETUNING_DATA_DIR, exist_ok=True)

# Vector DB setup
chroma_client = chromadb.PersistentClient(path=f"{SOURCE_TEXTS_DIR}chroma_db")

def make_llm_call(messages, model=None):
    """Make LLM call with OpenAI client."""
    model = model or MODEL_NAME
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

print("Configuration loaded.")
print(f"Project: {PROJECT_NAME}")
print(f"Model: {MODEL_NAME}")


Configuration loaded.
Project: QoGD
Model: gpt-5


---

## 📚 STEP 2: Load Selected Inspirations

Load the works you selected from the stylistic inspiration discovery step.

**What happens:** Loads `selected_inspirations.json` from the previous notebook.


In [11]:
# Load selected inspirations
def load_selected_inspirations():
    """Load selected inspirations from previous step."""
    if not os.path.exists(SELECTED_INSPIRATIONS_JSON):
        print(f"Warning: {SELECTED_INSPIRATIONS_JSON} not found.")
        print("Please run 1_stylistic_inspiration.ipynb first.")
        return []
    
    with open(SELECTED_INSPIRATIONS_JSON, 'r') as f:
        data = json.load(f)
    
    # Handle both formats: direct list or dictionary with 'selected_works' key
    if isinstance(data, list):
        inspirations = data
    elif isinstance(data, dict) and 'selected_works' in data:
        inspirations = data['selected_works']
    else:
        print(f"Warning: Unexpected format in {SELECTED_INSPIRATIONS_JSON}")
        return []
    
    print(f"Loaded {len(inspirations)} selected works:")
    for i, work in enumerate(inspirations, 1):
        print(f"{i}. {work['author']} - {work['title']}")
        print(f"   Reason: {work.get('reason', 'N/A')}")
    
    return inspirations

selected_works = load_selected_inspirations()


Loaded 7 selected works:
1. James Joyce - "The Dead" (in Dubliners)
   Reason: Irish funeral atmosphere, social texture, restrained irony, close interior epiphany; supports [LITURGY], [DRONE], [TIME-SLOW], and ambiguity around gendered roles embedded in communal scenes.
2. Anne Enright - The Gathering
   Reason: Modern Irish mourning and family memory; dissociative yet lucid POV; balances candor and ambiguity; strong template for funeral dialogue beats and interior drift.
3. Gerard Manley Hopkins - "The Blessed Virgin compared to the Air we Breathe" and other Marian poems
   Reason: Sprung rhythm echoes liturgy and Marian devotion; helps infuse reverent cadence without quoting rites; bridges to Mary’s numinous presence.
4. Flann O’Brien - The Third Policeman
   Reason: Metaphysical absurdity applied to ordinary scenes; identity dissolution and looping logic; supports “absurd descriptions of normal phenomena” and [NAUSEA] without losing readability.
5. Jean‑Paul Sartre - Nausea
   Reaso

---

## 🔧 STEP 3: Text Download Functions

Define helper functions for the agentic download process.

**Functions defined:**
- `find_text_urls_agentically()` - LLM finds concrete URLs for works
- `verify_text_content()` - LLM verifies downloaded text matches expected work
- `download_text_from_url()` - Downloads and extracts text from HTML
- `save_text_file()` - Saves verified text to local file


In [12]:
def find_text_urls_agentically(author: str, title: str) -> List[str]:
    """Use LLM to actively find URLs for the text online.
    
    Returns a list of concrete URLs to try, not just suggestions.
    """
    prompt = f"""I need to find the full text of "{title}" by {author} online.

Please provide SPECIFIC, CONCRETE URLs where this text is likely available. Do not provide search queries - provide actual URLs.

Common sources:
- Project Gutenberg: https://www.gutenberg.org/ebooks/[ID] or search: https://www.gutenberg.org/ebooks/search/?query=
- Internet Archive: https://archive.org/details/[identifier]
- Standard Ebooks: https://standardebooks.org/ebooks/[author-slug]/[title-slug]
- Wikisource: https://en.wikisource.org/wiki/[Title]

If you know the work, provide the specific URLs. If not, construct likely URLs based on the title and author.

Format your response as a JSON array of URL strings, like: ["url1", "url2", "url3"]
If you cannot provide URLs, respond with just: []"""

    messages = [
        {"role": "system", "content": "You are a helpful assistant that finds online sources for literary texts. Provide concrete URLs, not suggestions."},
        {"role": "user", "content": prompt}
    ]
    
    response = make_llm_call(messages)
    
    # Try to parse JSON from response
    try:
        # Look for JSON array in the response
        import json
        # Try to extract JSON array from response
        json_match = re.search(r'\[.*?\]', response, re.DOTALL)
        if json_match:
            urls = json.loads(json_match.group(0))
            return [url for url in urls if url.startswith('http')]
    except:
        pass
    
    # Fallback: extract URLs from text
    urls = re.findall(r'https?://[^\s\)]+', response)
    return urls[:5]  # Return up to 5 URLs

def verify_text_content(content: str, author: str, title: str) -> Dict[str, any]:
    """Use LLM to verify that downloaded content matches the expected work."""
    # Sample first and last 2000 chars
    sample = content[:2000] + "\n\n[... middle of text ...]\n\n" + content[-2000:] if len(content) > 4000 else content
    
    prompt = f"""I downloaded text that should be "{title}" by {author}. 

Please analyze this text sample and verify:
1. Is this actually the correct work? (Yes/No with confidence level)
2. What evidence do you see? (mentions of title, author, characteristic passages, etc.)
3. Is the text complete enough for our purposes? (full text, excerpt, or too short)
4. What percentage match confidence do you have? (0-100)

Text sample:
{sample[:3000]}

Respond in this format:
VERIFICATION: Yes/No
CONFIDENCE: [0-100]
EVIDENCE: [what you found]
COMPLETENESS: [full/excerpt/too short]
NOTES: [any additional observations]"""

    messages = [
        {"role": "system", "content": "You are a literary expert that verifies if downloaded text matches an expected work."},
        {"role": "user", "content": prompt}
    ]
    
    response = make_llm_call(messages)
    
    # Parse response
    verification = {
        'is_correct': False,
        'confidence': 0,
        'evidence': '',
        'completeness': 'unknown',
        'notes': '',
        'raw_response': response
    }
    
    # Extract information from response
    if 'VERIFICATION:' in response:
        verification['is_correct'] = 'yes' in response.split('VERIFICATION:')[1].split('\n')[0].lower()
    if 'CONFIDENCE:' in response:
        conf_match = re.search(r'CONFIDENCE:\s*(\d+)', response)
        if conf_match:
            verification['confidence'] = int(conf_match.group(1))
    if 'EVIDENCE:' in response:
        evidence_start = response.find('EVIDENCE:')
        evidence_end = response.find('\n', evidence_start + 20)
        if evidence_end == -1:
            evidence_end = len(response)
        verification['evidence'] = response[evidence_start+9:evidence_end].strip()
    if 'COMPLETENESS:' in response:
        compl_match = re.search(r'COMPLETENESS:\s*(\w+)', response)
        if compl_match:
            verification['completeness'] = compl_match.group(1).lower()
    if 'NOTES:' in response:
        notes_start = response.find('NOTES:')
        verification['notes'] = response[notes_start+6:].strip()
    
    return verification

def download_text_from_url(url: str) -> Optional[str]:
    """Download text from URL and extract content."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.decompose()
        
        # Try to find main content (Project Gutenberg, Archive.org patterns)
        content = None
        
        # Project Gutenberg
        if 'gutenberg.org' in url:
            content_elem = soup.find('div', class_='pg-boobstrip') or soup.find('div', id='content')
            if content_elem:
                content = content_elem.get_text()
            else:
                content = soup.get_text()
        
        # Archive.org
        elif 'archive.org' in url:
            content_elem = soup.find('div', class_='container-fluid') or soup.find('div', id='text-content')
            if content_elem:
                content = content_elem.get_text()
            else:
                content = soup.get_text()
        
        # Generic HTML
        else:
            content = soup.get_text()
        
        # Clean up text
        lines = content.split('\n')
        cleaned_lines = []
        for line in lines:
            line = line.strip()
            if line and len(line) > 10:  # Filter out very short lines
                cleaned_lines.append(line)
        
        return '\n\n'.join(cleaned_lines)
        
    except Exception as e:
        print(f"Error downloading from {url}: {e}")
        return None

def save_text_file(author: str, title: str, content: str):
    """Save downloaded text to file."""
    # Sanitize filename
    safe_author = re.sub(r'[^\w\s-]', '', author).strip()
    safe_title = re.sub(r'[^\w\s-]', '', title).strip()
    filename = f"{safe_author}_{safe_title}.txt"
    filepath = os.path.join(SOURCE_TEXTS_DIR, filename)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
    
    return filepath

# Note: Use the UI below to download texts agentically
# The system will find URLs, download, and verify the content automatically


---

## 📥 STEP 4: Download Source Texts

**🎯 Action Required:** Use the interface below to download your source texts.

### How It Works:
1. **Find URLs** - AI searches for where to find each work online
2. **Try Multiple URLs** - Downloads from several potential sources
3. **Verify Content** - AI checks if downloaded text matches expected work
4. **Select Best Match** - Chooses the highest-confidence verified text
5. **Save Locally** - Stores verified text for next steps

**💡 Tip:** Click **"Find & Download Automatically"** to let the AI handle everything. You can also manually paste URLs if needed.


In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

class TextDownloadUI:
    """UI for downloading texts from selected works."""
    
    def __init__(self, works: List[Dict]):
        self.works = works
        self.downloaded_texts = {}  # {work_index: filepath}
        self.current_work_index = 0
        
        # Widgets
        self.work_select = widgets.Dropdown(
            options=[(f"{w['author']} - {w['title']}", i) for i, w in enumerate(works)],
            description="Work:",
            value=0
        )
        
        self.url_input = widgets.Text(
            value="",
            description="URL:",
            placeholder="Paste URL or leave empty for LLM suggestions",
            layout=widgets.Layout(width="600px")
        )
        
        self.auto_download_button = widgets.Button(
            description="Find & Download Automatically",
            button_style="primary",
            layout=widgets.Layout(width="250px")
        )
        
        self.download_button = widgets.Button(
            description="Download from URL",
            button_style="success",
            layout=widgets.Layout(width="200px")
        )
        
        self.status_display = widgets.HTML(
            value="<p>Ready to download texts.</p>",
            layout=widgets.Layout(width="100%", min_height="100px")
        )
        
        self.output = widgets.Output()
        
        # Event handlers
        self.work_select.observe(self.on_work_change, names='value')
        self.auto_download_button.on_click(self.find_and_download_agentically)
        self.download_button.on_click(self.download_text)
        
    def display(self):
        """Display the UI."""
        display(widgets.VBox([
            widgets.HTML("<h3>Download Source Texts</h3>"),
            widgets.HTML("<p><strong>Recommended:</strong> Use 'Find & Download Automatically' to let the AI find URLs, download, and verify the text.</p>"),
            self.work_select,
            self.auto_download_button,
            widgets.HTML("<hr><p><strong>Or manually:</strong></p>"),
            widgets.HBox([
                self.url_input,
                self.download_button
            ]),
            self.status_display,
            self.output
        ]))
        
    def on_work_change(self, change):
        """Handle work selection change."""
        work_idx = change['new']
        work = self.works[work_idx]
        
        if work_idx in self.downloaded_texts:
            status = f"<p style='color: green;'>✓ Already downloaded: {self.downloaded_texts[work_idx]}</p>"
        else:
            status = f"<p>Ready to download: <strong>{work['author']} - {work['title']}</strong></p>"
        
        self.status_display.value = status
        self.url_input.value = ""
        
    def find_and_download_agentically(self, _):
        """Agentically find URLs, download, and verify the text."""
        work_idx = self.work_select.value
        work = self.works[work_idx]
        
        with self.output:
            clear_output()
            print(f"Finding URLs for {work['author']} - {work['title']}...")
            print("Step 1: Getting URLs from LLM...")
        
        # Step 1: Find URLs agentically
        urls = find_text_urls_agentically(work['author'], work['title'])
        
        if not urls:
            self.status_display.value = "<p style='color: orange;'>⚠ No URLs found. Try manual URL entry.</p>"
            with self.output:
                clear_output()
                print("⚠ No URLs found by LLM")
            self.url_input.value = "No URLs found. Please enter manually."
            return
        
        with self.output:
            clear_output()
            print(f"✓ Found {len(urls)} URLs to try:")
            for i, url in enumerate(urls, 1):
                print(f"  {i}. {url}")
            print(f"\nStep 2: Downloading and verifying from each URL...")
        
        self.status_display.value = f"<p>Trying {len(urls)} URLs...</p>"
        
        # Step 2: Try each URL, download, and verify
        best_match = None
        best_verification = None
        best_url = None
        
        for i, url in enumerate(urls, 1):
            with self.output:
                print(f"\nTrying URL {i}/{len(urls)}: {url}")
            
            try:
                content = download_text_from_url(url)
                
                if not content or len(content) < 100:
                    with self.output:
                        print(f"  ✗ Download failed or too short")
                    continue
                
                with self.output:
                    print(f"  ✓ Downloaded ({len(content)} chars), verifying...")
                
                # Step 3: Verify the content
                verification = verify_text_content(content, work['author'], work['title'])
                
                with self.output:
                    print(f"  Verification: {'✓ CORRECT' if verification['is_correct'] else '✗ INCORRECT'}")
                    print(f"    Confidence: {verification['confidence']}%")
                    print(f"    Completeness: {verification['completeness']}")
                    if verification['evidence']:
                        print(f"    Evidence: {verification['evidence'][:100]}...")
                
                # Keep track of best match
                if verification['is_correct']:
                    if not best_match or verification['confidence'] > best_verification['confidence']:
                        best_match = content
                        best_verification = verification
                        best_url = url
                elif not best_match and verification['confidence'] > 50:
                    # If no correct match yet, keep high-confidence incorrect matches as fallback
                    if not best_match or verification['confidence'] > best_verification['confidence']:
                        best_match = content
                        best_verification = verification
                        best_url = url
                        
            except Exception as e:
                with self.output:
                    print(f"  ✗ Error: {e}")
                continue
        
        # Step 4: Save best match
        if best_match:
            filepath = save_text_file(work['author'], work['title'], best_match)
            self.downloaded_texts[work_idx] = filepath
            
            status_color = "green" if best_verification['is_correct'] else "orange"
            status_icon = "✓" if best_verification['is_correct'] else "⚠"
            
            status_html = f"""<p style='color: {status_color};'>
                {status_icon} Downloaded from: <a href="{best_url}" target="_blank">{best_url}</a><br>
                Verification: {'CORRECT' if best_verification['is_correct'] else 'UNCERTAIN'} ({best_verification['confidence']}% confidence)<br>
                Completeness: {best_verification['completeness']}<br>
                Saved to: {filepath}
            </p>"""
            
            if best_verification['evidence']:
                status_html += f"<p><small>Evidence: {best_verification['evidence']}</small></p>"
            
            self.status_display.value = status_html
            
            with self.output:
                print(f"\n{'='*60}")
                print(f"✓ Saved to {filepath}")
                print(f"Verification: {best_verification['is_correct']} ({best_verification['confidence']}% confidence)")
                print(f"{'='*60}")
        else:
            self.status_display.value = "<p style='color: red;'>✗ No suitable text found. Try different URLs or manual download.</p>"
            with self.output:
                print("\n✗ No suitable text found from any URL")
    
    def download_text(self, _):
        """Download text from manually entered URL."""
        work_idx = self.work_select.value
        work = self.works[work_idx]
        url = self.url_input.value.strip()
        
        if not url or url.startswith("No URLs"):
            self.status_display.value = "<p style='color: orange;'>⚠ Please click 'Find & Download Automatically' or enter a URL manually.</p>"
            return
        
        with self.output:
            clear_output()
            print(f"Downloading from {url}...")
        
        content = download_text_from_url(url)
        
        if content and len(content) > 100:
            # Verify the content
            with self.output:
                print("Verifying downloaded content...")
            
            verification = verify_text_content(content, work['author'], work['title'])
            
            with self.output:
                print(f"Verification: {'✓ CORRECT' if verification['is_correct'] else '✗ INCORRECT'}")
                print(f"Confidence: {verification['confidence']}%")
                print(f"Evidence: {verification['evidence']}")
            
            filepath = save_text_file(work['author'], work['title'], content)
            self.downloaded_texts[work_idx] = filepath
            
            status_color = "green" if verification['is_correct'] else "orange"
            status_icon = "✓" if verification['is_correct'] else "⚠"
            
            status_html = f"""<p style='color: {status_color};'>
                {status_icon} Downloaded ({len(content)} chars)<br>
                Verification: {'CORRECT' if verification['is_correct'] else 'UNCERTAIN'} ({verification['confidence']}% confidence)<br>
                Saved to: {filepath}
            </p>"""
            
            self.status_display.value = status_html
        else:
            self.status_display.value = "<p style='color: red;'>✗ Download failed. Check URL or try different source.</p>"
            with self.output:
                print("✗ Download failed or content too short")

if selected_works:
    download_ui = TextDownloadUI(selected_works)
    download_ui.display()
else:
    print("No selected works to download. Run 1_stylistic_inspiration.ipynb first.")


---

## 🔍 STEP 5: Create Vector Databases

**What happens:** Processes all downloaded texts into searchable vector embeddings.

This step:
- Splits each text into paragraphs
- Generates embeddings (using OpenAI `text-embedding-3-small`)
- Stores in ChromaDB for fast semantic search
- Allows finding sections by description (e.g., "opening paragraph about the sea")

**⏱️ Note:** This may take a few minutes depending on text size. Progress bars will show.


In [14]:
def create_vector_db_for_text(filepath: str, author: str, title: str):
    """Create vector database for a text file."""
    # Read text
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # Split into paragraphs
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip() and len(p.strip()) > 50]
    
    # Create collection name (unique per text)
    collection_name = hashlib.md5(f"{author}_{title}".encode()).hexdigest()[:16]
    
    # Get or create collection
    try:
        collection = chroma_client.get_collection(collection_name)
        print(f"Using existing collection: {collection_name}")
    except:
        collection = chroma_client.create_collection(collection_name)
        print(f"Created new collection: {collection_name}")
        
        # Generate embeddings and add to collection
        print(f"Processing {len(paragraphs)} paragraphs...")
        for i, para in enumerate(tqdm(paragraphs)):
            # Use OpenAI embeddings
            embedding_response = openai_client.embeddings.create(
                model="text-embedding-3-small",
                input=para
            )
            embedding = embedding_response.data[0].embedding
            
            # Add to collection
            collection.add(
                embeddings=[embedding],
                documents=[para],
                ids=[f"para_{i}"]
            )
        
        print(f"✓ Added {len(paragraphs)} paragraphs to vector DB")
    
    return collection, paragraphs

def load_outline():
    """Load outline for context."""
    possible_files = [
        f"{SOURCE_PROMPTS_DIR}outline.txt",
        f"{SOURCE_PROMPTS_DIR}write_prompt_1.txt",
    ]
    for file_path in possible_files:
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                return f.read()
    return None

# Process downloaded texts and create vector DBs
print("Processing downloaded texts...")
outline_text = load_outline() or ""

text_collections = {}  # {work_index: (collection, paragraphs)}

for work_idx, work in enumerate(selected_works):
    # Check if file exists
    safe_author = re.sub(r'[^\w\s-]', '', work['author']).strip()
    safe_title = re.sub(r'[^\w\s-]', '', work['title']).strip()
    filename = f"{safe_author}_{safe_title}.txt"
    filepath = os.path.join(SOURCE_TEXTS_DIR, filename)
    
    if os.path.exists(filepath):
        print(f"\nProcessing {work['author']} - {work['title']}...")
        collection, paragraphs = create_vector_db_for_text(filepath, work['author'], work['title'])
        text_collections[work_idx] = (collection, paragraphs, filepath)
    else:
        print(f"Skipping {work['author']} - {work['title']} (file not found)")

print(f"\n✓ Processed {len(text_collections)} texts")


Processing downloaded texts...

Processing James Joyce - "The Dead" (in Dubliners)...
Created new collection: 54f16b55165ee8b8
Processing 357 paragraphs...


100%|████████████████████████████████████████████████████████████████████████████████████████████| 357/357 [01:39<00:00,  3.57it/s]

✓ Added 357 paragraphs to vector DB
Skipping Anne Enright - The Gathering (file not found)
Skipping Gerard Manley Hopkins - "The Blessed Virgin compared to the Air we Breathe" and other Marian poems (file not found)
Skipping Flann O’Brien - The Third Policeman (file not found)
Skipping Jean‑Paul Sartre - Nausea (file not found)
Skipping Thomas Bernhard - Concrete (or Woodcutters) (file not found)
Skipping Clarice Lispector - Água Viva (file not found)

✓ Processed 1 texts


---

## 🤖 STEP 6: Get Section Recommendations

**What happens:** AI analyzes each text and recommends relevant sections for fine-tuning.

### Two-Step AI Process:

**Step A - Knowledge-Based Guidance:**
- LLM uses its knowledge of the work to identify what types of sections would be useful
- Considers your story outline to match style/themes
- Provides guidance on what to look for

**Step B - Text-Based Search:**
- LLM searches the actual downloaded text
- Finds sections matching the guidance
- Provides 5-10 specific recommendations with quotes

**⏱️ Note:** This step makes LLM calls for each work, so it may take a few minutes.


In [15]:
def get_section_recommendations(text_content: str, outline: str, author: str, title: str) -> str:
    """Get LLM recommendations for sections to use for fine-tuning.
    
    Uses a two-step approach:
    1. First asks LLM for general guidance based on its knowledge
    2. Then uses that guidance to search through the actual text
    """
    # Step 1: Get general guidance from LLM's knowledge
    guidance_prompt = f"""Based on your knowledge of "{title}" by {author}, and this story outline, what kinds of sections from this work would be most stylistically useful for fine-tuning a writing model?

Story Outline:
{outline[:1500] if len(outline) > 1500 else outline}

Please provide guidance on:
1. What types of passages to look for (e.g., "opening paragraphs", "dialogue scenes", "descriptive passages", "introspective moments")
2. What themes, stylistic elements, or techniques to prioritize
3. What specific chapters, scenes, or sections are known to be particularly noteworthy

Provide 5-10 specific types of sections to search for, with explanations of why each would be relevant."""

    guidance_messages = [
        {"role": "system", "content": "You are a literary advisor with deep knowledge of literature. Help identify what types of sections would be most useful for stylistic fine-tuning."},
        {"role": "user", "content": guidance_prompt}
    ]
    
    print("  Step 1: Getting general guidance from LLM knowledge...")
    guidance = make_llm_call(guidance_messages)
    print("  ✓ Got guidance")
    
    # Step 2: Use guidance to search through the actual text
    # Use a sample of the text (first 10000 chars, middle 10000 chars, last 10000 chars if available)
    text_sample = text_content[:10000]
    if len(text_content) > 20000:
        mid_start = len(text_content) // 2 - 5000
        mid_end = len(text_content) // 2 + 5000
        text_sample += f"\n\n[... middle section ...]\n\n{text_content[mid_start:mid_end]}"
    if len(text_content) > 10000:
        text_sample += f"\n\n[... end section ...]\n\n{text_content[-10000:]}"
    
    search_prompt = f"""Now I have actual text from "{title}" by {author}. Based on the guidance below, find specific sections in this text that match the criteria.

Guidance from your knowledge:
{guidance}

Text sample (various sections from the work):
{text_sample}

Please find 5-10 specific sections from this text that match the guidance above. For each section you identify, provide:
1. A brief description of what you found (e.g., "opening paragraph that matches the introspective style", "dialogue scene with characteristic voice")
2. Why this section matches the guidance (which criteria it fulfills)
3. A unique quote or phrase from the section that can be used to identify it (at least 10 words)

Format your response as a numbered list with clear markers for description, reason, and quote."""

    search_messages = [
        {"role": "system", "content": "You are a literary advisor helping locate specific passages in a text based on guidance about what to look for."},
        {"role": "user", "content": search_prompt}
    ]
    
    print("  Step 2: Searching actual text based on guidance...")
    recommendations = make_llm_call(search_messages)
    print("  ✓ Found sections")
    
    # Combine guidance and recommendations
    combined = f"""# Guidance Based on LLM Knowledge:

{guidance}

---

# Recommended Sections Found in Text:

{recommendations}"""
    
    return combined

# Get recommendations for each text
recommendations = {}  # {work_idx: recommendations_text}

for work_idx in text_collections.keys():
    collection, paragraphs, filepath = text_collections[work_idx]
    work = selected_works[work_idx]
    
    # Read full text for recommendations
    with open(filepath, 'r', encoding='utf-8') as f:
        full_text = f.read()
    
    print(f"\nGetting recommendations for {work['author']} - {work['title']}...")
    recs = get_section_recommendations(full_text, outline_text, work['author'], work['title'])
    recommendations[work_idx] = recs
    print(f"✓ Got recommendations")
    print(recs[:300] + "..." if len(recs) > 300 else recs)



Getting recommendations for James Joyce - "The Dead" (in Dubliners)...
  Step 1: Getting general guidance from LLM knowledge...
  ✓ Got guidance
  Step 2: Searching actual text based on guidance...
  ✓ Found sections
✓ Got recommendations
# Guidance Based on LLM Knowledge:

Below are the most useful kinds of passages in The Dead to fine‑tune a model toward your outline’s tone, themes, and techniques. Each item names the passage type, points you to a specific part of the story, and explains why it maps well to your goals.

1) Threshol...


---

## ✂️ STEP 7: Select Sections for Fine-tuning

**🎯 Action Required:** Review and select which recommended sections to include in your dataset.

### How to Use:

1. **Browse Recommendations** - See all AI-recommended sections on the left
2. **View Section Text** - Read the full section text in the editor
3. **Adjust Boundaries** - Use ↑/↓ buttons to expand or contract selections
4. **Navigate** - Use Previous/Next to move between sections
5. **Add to Dataset** - Click "Add to Dataset" for sections you want to keep
6. **Skip** - Click "Skip" to move on without adding
7. **Save** - Click "Save All Sections" when done with all works

### Tips:
- You can see the LLM's guidance (why sections were recommended) by expanding the accordion
- Selected sections are tracked at the bottom
- All sections will be saved to `finetuning_data/{PROJECT_NAME}/combined_act2.txt`


In [ ]:
def find_section_in_vector_db(collection, description: str, paragraphs: List[str], context_lines: int = 2) -> Optional[Dict]:
    """Find a section in vector DB based on description."""
    # Get embedding for description
    embedding_response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=description
    )
    embedding = embedding_response.data[0].embedding
    
    # Query vector DB
    results = collection.query(
        query_embeddings=[embedding],
        n_results=1
    )
    
    if results['documents'] and len(results['documents'][0]) > 0:
        found_para = results['documents'][0][0]
        para_id = results['ids'][0][0]
        para_idx = int(para_id.split('_')[1])
        
        # Get context (surrounding paragraphs)
        start_idx = max(0, para_idx - context_lines)
        end_idx = min(len(paragraphs), para_idx + context_lines + 1)
        context = '\n\n'.join(paragraphs[start_idx:end_idx])
        
        return {
            'matched_paragraph': found_para,
            'context': context,
            'paragraph_index': para_idx,
            'start_context_idx': start_idx,
            'end_context_idx': end_idx
        }
    
    return None

class SectionSelectionUI:
    """Interactive UI for selecting sections from recommended passages."""
    
    def __init__(self, works, text_collections, recommendations):
        self.works = works
        self.text_collections = text_collections
        self.recommendations = recommendations
        self.selected_sections = []  # List of {work_idx, section_text, metadata}
        
        self.current_work_idx = list(text_collections.keys())[0] if text_collections else None
        self.current_recommendations_text = recommendations.get(self.current_work_idx, "") if self.current_work_idx else ""
        self.current_section_idx = 0
        
        # Parse recommendations into sections
        self.parsed_sections = self._parse_recommendations(self.current_recommendations_text) if self.current_recommendations_text else []
        
        # Widgets
        self.work_select = widgets.Dropdown(
            options=[(f"{works[i]['author']} - {works[i]['title']}", i) for i in text_collections.keys()],
            description="Work:",
            value=self.current_work_idx
        )
        
        self.recommendations_display = widgets.HTML(
            value=self._format_recommendations(self.parsed_sections),
            layout=widgets.Layout(width="100%", height="300px", overflow="auto")
        )
        
        self.guidance_display = widgets.HTML(
            value="",
            layout=widgets.Layout(width="100%", max_height="200px", overflow="auto")
        )
        
        self.section_text_display = widgets.Textarea(
            value="",
            description="Section text:",
            layout=widgets.Layout(width="100%", height="400px")
        )
        
        self.expand_up_button = widgets.Button(description="↑ Expand Up", button_style="")
        self.expand_down_button = widgets.Button(description="↓ Expand Down", button_style="")
        self.skip_button = widgets.Button(description="Skip", button_style="")
        self.add_button = widgets.Button(description="Add to Dataset", button_style="success")
        
        self.nav_prev = widgets.Button(description="← Previous", button_style="")
        self.nav_next = widgets.Button(description="Next →", button_style="")
        self.section_counter = widgets.Label(value="Section 0/0")
        
        self.status = widgets.HTML(value="<p>Ready to select sections.</p>")
        self.output = widgets.Output()
        
        # Event handlers
        self.work_select.observe(self.on_work_change, names='value')
        self.recommendations_display.observe(self.on_recommendation_click, names='value')
        self.expand_up_button.on_click(self.expand_up)
        self.expand_down_button.on_click(self.expand_down)
        self.skip_button.on_click(self.skip_section)
        self.add_button.on_click(self.add_section)
        self.nav_prev.on_click(self.prev_section)
        self.nav_next.on_click(self.next_section)
        
        # Load first section
        self.load_current_section()
        
    def _parse_recommendations(self, recs_text: str) -> List[Dict]:
        """Parse LLM recommendations into structured sections.
        
        Handles the new two-part format with guidance and recommendations.
        Only parses the recommendations section.
        """
        # Extract only the recommendations part (after the separator)
        lines = recs_text.split('\n')
        in_recommendations_section = False
        recommendations_lines = []
        
        for line in lines:
            # Look for the recommendations section header
            if 'Recommended Sections Found in Text' in line or 'Recommended Sections' in line:
                in_recommendations_section = True
                continue
            # Skip guidance section (before separator or header)
            if 'Guidance Based on LLM Knowledge' in line or (not in_recommendations_section and '---' not in line):
                if '---' in line and not in_recommendations_section:
                    in_recommendations_section = True
                continue
            # Collect recommendations lines
            if in_recommendations_section:
                recommendations_lines.append(line)
        
        # If no separator found, assume entire text is recommendations (backwards compatibility)
        if not recommendations_lines:
            recommendations_lines = lines
        
        # Parse recommendations
        sections = []
        current_section = None
        
        for line in recommendations_lines:
            line = line.strip()
            if not line:
                continue
            
            # Check if this is a numbered item
            match = re.match(r'^(\d+)\.\s*(.+)', line)
            if match:
                if current_section:
                    sections.append(current_section)
                current_section = {
                    'number': int(match.group(1)),
                    'description': match.group(2),
                    'reason': '',
                    'quote': ''
                }
            elif current_section:
                # Check for "Why:" or "Reason:" markers
                if re.match(r'^(Why|Reason|Relevant|matches|fulfills):\s*(.+)', line, re.I):
                    match_obj = re.match(r'^(Why|Reason|Relevant|matches|fulfills):\s*(.+)', line, re.I)
                    current_section['reason'] = match_obj.group(2)
                # Check for "Quote:" markers
                elif re.match(r'^(Quote|Phrase):\s*(.+)', line, re.I):
                    match_obj = re.match(r'^(Quote|Phrase):\s*(.+)', line, re.I)
                    current_section['quote'] = match_obj.group(2)
                else:
                    # Append to description or reason
                    if not current_section['reason']:
                        current_section['description'] += " " + line
                    else:
                        current_section['reason'] += " " + line
        
        if current_section:
            sections.append(current_section)
        
        return sections
    
    def _format_recommendations(self, sections: List[Dict]) -> str:
        """Format recommendations for display."""
        html = "<div>"
        for i, section in enumerate(sections):
            selected_class = "style='background-color: #e3f2fd;'" if i == self.current_section_idx else ""
            html += f"<div {selected_class} style='padding: 10px; margin: 5px; border: 1px solid #ccc; cursor: pointer;'>"
            html += f"<strong>{section['number']}. {section['description']}</strong>"
            if section.get('reason'):
                html += f"<br><small>{section['reason']}</small>"
            html += "</div>"
        html += "</div>"
        return html
    
    def on_work_change(self, change):
        """Handle work selection change."""
        self.current_work_idx = change['new']
        self.current_recommendations_text = self.recommendations.get(self.current_work_idx, "")
        self.parsed_sections = self._parse_recommendations(self.current_recommendations_text) if self.current_recommendations_text else []
        self.current_section_idx = 0
        self.recommendations_display.value = self._format_recommendations(self.parsed_sections)
        # Update guidance display
        guidance_text = self._extract_guidance(self.current_recommendations_text)
        self.guidance_display.value = f"<div style='white-space: pre-wrap;'>{guidance_text}</div>" if guidance_text else ""
        self.load_current_section()
    
    def on_recommendation_click(self, change):
        """Handle clicking on a recommendation (placeholder - would need JS for full implementation)."""
        pass
    
    def load_current_section(self):
        """Load the current section text."""
        if not self.parsed_sections or self.current_section_idx >= len(self.parsed_sections):
            self.section_text_display.value = "No more sections."
            return
        
        section = self.parsed_sections[self.current_section_idx]
        collection, paragraphs, filepath = self.text_collections[self.current_work_idx]
        
        # Try to find section using description or quote
        search_text = section.get('quote') or section['description']
        result = find_section_in_vector_db(collection, search_text, paragraphs)
        
        if result:
            self.section_text_display.value = result['context']
            self.current_section_data = {
                'result': result,
                'section': section,
                'work_idx': self.current_work_idx
            }
        else:
            self.section_text_display.value = f"Could not find section. Description: {section['description']}"
            self.current_section_data = None
        
        self.section_counter.value = f"Section {self.current_section_idx + 1}/{len(self.parsed_sections)}"
        self.recommendations_display.value = self._format_recommendations(self.parsed_sections)
    
    def expand_up(self, _):
        """Expand selection upward."""
        if not hasattr(self, 'current_section_data') or not self.current_section_data:
            return
        
        result = self.current_section_data['result']
        _, paragraphs, _ = self.text_collections[self.current_work_idx]
        
        if result['start_context_idx'] > 0:
            result['start_context_idx'] -= 1
            result['context'] = '\n\n'.join(paragraphs[result['start_context_idx']:result['end_context_idx']])
            self.section_text_display.value = result['context']
    
    def expand_down(self, _):
        """Expand selection downward."""
        if not hasattr(self, 'current_section_data') or not self.current_section_data:
            return
        
        result = self.current_section_data['result']
        _, paragraphs, _ = self.text_collections[self.current_work_idx]
        
        if result['end_context_idx'] < len(paragraphs):
            result['end_context_idx'] += 1
            result['context'] = '\n\n'.join(paragraphs[result['start_context_idx']:result['end_context_idx']])
            self.section_text_display.value = result['context']
    
    def skip_section(self, _):
        """Skip current section."""
        self.next_section(None)
    
    def add_section(self, _):
        """Add current section to dataset."""
        if not hasattr(self, 'current_section_data') or not self.current_section_data:
            with self.output:
                clear_output()
                print("No section loaded.")
            return
        
        section_text = self.section_text_display.value
        work = self.works[self.current_work_idx]
        
        self.selected_sections.append({
            'work_author': work['author'],
            'work_title': work['title'],
            'section_text': section_text,
            'section_description': self.current_section_data['section']['description'],
            'reason': self.current_section_data['section'].get('reason', '')
        })
        
        with self.output:
            clear_output()
            print(f"✓ Added section from {work['author']} - {work['title']}")
            print(f"Total sections: {len(self.selected_sections)}")
        
        self.status.value = f"<p style='color: green;'>✓ Added section ({len(self.selected_sections)} total)</p>"
        
        # Move to next section
        self.next_section(None)
    
    def prev_section(self, _):
        """Go to previous section."""
        if self.current_section_idx > 0:
            self.current_section_idx -= 1
            self.load_current_section()
    
    def next_section(self, _):
        """Go to next section."""
        if self.current_section_idx < len(self.parsed_sections) - 1:
            self.current_section_idx += 1
            self.load_current_section()
        else:
            with self.output:
                clear_output()
                print("Reached end of recommendations for this work.")
    
    def save_sections(self, _=None):
        """Save selected sections to fine-tuning data file."""
        if not self.selected_sections:
            with self.output:
                clear_output()
                print("No sections to save.")
            return False
        
        # Append to output file (matching format from 3_finetune-e2e-fiction.ipynb)
        with open(OUTPUT_DATA_FILE, 'a', encoding='utf-8') as f:
            for section in self.selected_sections:
                f.write("---\n")  # Separator
                f.write(f"# From: {section['work_author']} - {section['work_title']}\n")
                f.write(f"# {section['section_description']}\n")
                f.write(f"{section['section_text']}\n\n")
        
        self.status.value = f"<p style='color: green;'>✓ Saved {len(self.selected_sections)} sections to {OUTPUT_DATA_FILE}</p>"
        with self.output:
            clear_output()
            print(f"✓ Saved {len(self.selected_sections)} sections to {OUTPUT_DATA_FILE}")
        return True
    
    def _create_save_button(self):
        """Create save button with proper handler."""
        save_btn = widgets.Button(
            description="Save All Sections to Dataset",
            button_style="warning",
            layout=widgets.Layout(width="300px")
        )
        save_btn.on_click(self.save_sections)
        return save_btn
    
    def _extract_guidance(self, recommendations_text: str) -> str:
        """Extract guidance section from combined recommendations text."""
        if '---' in recommendations_text:
            guidance = recommendations_text.split('---')[0]
            # Remove header if present
            if 'Guidance Based on LLM Knowledge' in guidance:
                guidance = guidance.split('Guidance Based on LLM Knowledge')[1].strip()
                guidance = guidance.lstrip('#').strip()
            return guidance
        return ""
    
    def display(self):
        """Display the UI."""
        # Add guidance display (collapsible)
        guidance_text = self._extract_guidance(self.current_recommendations_text)
        self.guidance_display.value = f"<div style='white-space: pre-wrap;'>{guidance_text}</div>" if guidance_text else ""
        guidance_accordion = widgets.Accordion(children=[self.guidance_display], selected_index=None, titles=('Show Guidance from LLM Knowledge',))
        
        display(widgets.VBox([
            widgets.HTML("<h3>Select Sections for Fine-tuning</h3>"),
            self.work_select,
            guidance_accordion,
            widgets.HBox([
                widgets.VBox([
                    widgets.HTML("<h4>Recommended Sections</h4>"),
                    self.recommendations_display,
                ], layout=widgets.Layout(width="40%")),
                widgets.VBox([
                    widgets.HBox([self.section_counter, self.nav_prev, self.nav_next]),
                    self.section_text_display,
                    widgets.HBox([
                        self.expand_up_button,
                        self.expand_down_button,
                        self.skip_button,
                        self.add_button
                    ]),
                    self.status,
                ], layout=widgets.Layout(width="60%")),
            ]),
            widgets.HTML(f"<p><strong>Selected sections: {len(self.selected_sections)}</strong></p>"),
            self._create_save_button(),
            self.output
        ]))

if text_collections:
    selection_ui = SectionSelectionUI(selected_works, text_collections, recommendations)
    selection_ui.display()
else:
    print("No texts processed. Download texts first.")
